# Gold Layer Incremental
Incremental Gold Processing plus latest and timestamped Volume snapshots



## Step 1 -- Imports and Setup

This cell imports the required helpers, switches to the right catalog, makes sure the **Gold Schema** exists and creates:

- `gold_run_id`
- a run date string
- a run timestamp string


These are used for tracking and snapshot publishing 

In [0]:
import pyspark.sql.functions as f
from delta.tables import DeltaTable
from datetime import datetime
import uuid

In [0]:
spark.sql("use catalog ecomdatabricks")
spark.sql("create schema if not exists gold_schema")
gold_run_id = str(uuid.uuid4())


run_ts_str = datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S')
run_date_str = datetime.utcnow().strftime('%Y-%m-%d')

print("Current Gold run id: ", gold_run_id)
print("Run Timestamp Folder: ", run_ts_str)
# DBTITLE 1

## Step 2 -- Gold control table
This table stores the latest processing state

It tells Gold:
- Which Silver data was processed last time
- How many Gold rows were merged in the last run

In [0]:
spark.sql("""
            create table if not exists ecomdatabricks.gold_schema.processing_control(
                layer string,
                entity_name string,
                last_processed_silver_run_id string,
                last_processed_silver_run_ts timestamp,
                rows_merged bigint,
                run_status string,
                gold_run_id string,
                updated_at timestamp
            )
            using delta
          
          """)

## Step 3. -- Helper Functions
This cells defines reusable Gold functions.
- `upsert_to_gold()` merges data into Gold current-state tables.
- `get_last_processed_silver_ts()` reads the Gold watermark from the control table.
- `upsert_gold_control()` updates Gold control after a successful run.

In [0]:
def upsert_to_gold(df_source, target_table, join_key):
  if spark.catalog.tableExists(target_table):
    dt = DeltaTable.forName(spark, target_table)
    (dt.alias("target")
      .merge(df_source.alias("source"), f"target.{join_key} = source.{join_key}")
      .whenMatchedUpdateAll()
      .whenNotMatchedInsertAll()
      .execute())
  else:
    df_source.write.format("delta").saveAsTable(target_table)


In [0]:
def get_last_processed_silver_ts(entity_name:str):
    ctrl = (
        spark.table("ecomdatabricks.gold_schema.processing_control")
        .filter(
            (f.col("layer")=="gold") &
            (f.col("entity_name")==entity_name) &
            (f.col("run_status")=="SUCCESS")
        )
        .orderBy(f.col("updated_at").desc())
        .limit(1)
        
    )

    rows = ctrl.collect()

    if not rows:
        return None
    else:
        rows[0]["last_processed_silver_run_ts"]

In [0]:
def upsert_gold_control(entity_name, last_processed_silver_run_id, last_processed_run_ts, rows_merged):
    ctrl_df = spark.createDataFrame(
        [(
            "gold",
            entity_name,
            last_processed_silver_run_id,
            last_processed_run_ts,
            rows_merged,
            "SUCCESS",
            gold_run_id,
            datetime.utcnow()
        )],
        schema="""
            layer string,
            entity_name string,
            last_processed_silver_run_id string,
            last_processed_silver_run_ts timestamp,
            rows_merged bigint,
            run_status string,
            gold_run_id string,
            updated_at timestamp
        """
    )

    dt = DeltaTable.forName(spark, "ecomdatabricks.gold_schema.processing_control")
    (dt.alias("t")
      .merge(ctrl_df.alias("s"), "t.layer=s.layer and t.entity_name = s.entity_name")
      .whenMatchedUpdate(set={
          "last_processed_silver_run_id": "s.last_processed_silver_run_id",
          "last_processed_silver_run_ts": "s.last_processed_silver_run_ts",
          "rows_merged": "s.rows_merged",
          "run_status": "s.run_status",
          "gold_run_id": "s.gold_run_id",
          "updated_at": "s.updated_at"
      })
      .whenNotMatchedInsertAll()
      .execute()
    )

## Step 4 -- Read changed Silver rows only
This cell reads the full Silver current-state tables but filters out only the rows that chnaged since the last Gold run
This is the starting point for Gold incremental processing

In [0]:
last_gold_ts = get_last_processed_silver_ts("orders_information")

print("Last Processed Silver Timestamp for Gold ", last_gold_ts)


silver_orders_current = spark.read.table("ecomdatabricks.silver_schema.orders_transformed")
silver_products_current = spark.read.table("ecomdatabricks.silver_schema.products_transformed")
silver_payments_current = spark.read.table("ecomdatabricks.silver_schema.payments_transformed")

if last_gold_ts is None:

  changed_orders = silver_orders_current
  changed_products = silver_products_current
  changed_payments = silver_payments_current

else:

  changed_orders = silver_orders_current.filter(f.col("updated_at") > f.col(last_gold_ts))
  changed_products = silver_products_current.filter(f.col("updated_at") > f.col(last_gold_ts))
  changed_payments = silver_payments_current.filter(f.col("updated_at") > f.col(last_gold_ts))


changed_orders_count = changed_orders.count()
changed_products_count = changed_products.count()
changed_payments_count = changed_payments.count()

print("Number of changed Orders : ", changed_orders_count)
print("Number of changed Products : ", changed_products_count)
print("Number of changed Payments : ", changed_payments_count)



## Step 5 -- Find the Impacted order IDs

Goild is built at **order grain**, so if anything changes in the orders,products or payments we identfy  which `order_id` values are impacted.
Only those order IDs are rebuilt in Gold


In [0]:
impacted_from_orders = changed_orders.select("order_id").distinct()
impacted_from_payments = changed_payments.select("order_id").distinct()

impacted_from_products = (
  changed_products.alias("p")
  .join(silver_orders_current.alias("o"),
        f.col("p.product_id")==f.col("o.product_id"),
        "inner"
  )
  .select(f.col("o.order_id")).distinct()
)

impacted_order_ids = (
  impacted_from_orders
  .union(impacted_from_payments)
  .union(impacted_from_products)
  .distinct()
)

print("Impacted Order IDs ", impacted_order_ids.count())
display(impacted_order_ids.orderBy("order_id"))

## Step 6 -- Build. Gold delta for impacted orders

This cell joins the impacted orders with the current Silver Products and payments table, derives business columns and builds **Gold delta** that will be merged into the Gold current-state table. 

In [0]:
impacted_orders = (
    silver_orders_current.alias("o")
    .join(impacted_order_ids.alias("i"), "order_id", "inner")
)

gold_delta = (
    impacted_orders.alias("o")
    .join(
        silver_products_current.alias("p"),
        f.col("o.product_id")==f.col("p.product_id"),
        "inner"
    )
    .join(
        silver_payments_current.alias("pay"),
        f.col("o.order_id")==f.col("pay.order_id")
    )
    .select(
        f.col("o.order_id"),
        f.col("o.customer_id"),
        f.col("p.product_id"),
        f.col("p.product_name"),
        f.col("p.category"),
        f.col("p.price").alias("product_price"),
        f.col("o.order_status"),
        f.col("o.order_amount"),
        f.col("pay.payment_id"),
        f.col("pay.payment_status"),
        f.col("pay.paid_amount"),
        f.col("o.order_date"),
        f.col("o.order_month"),
        f.col("o.order_year"),
        f.greatest(
            f.col("o.updated_at").cast("timestamp"),
            f.col("p.updated_at").cast("timestamp"),
            f.col("pay.processed_at").cast("timestamp")
        ).alias("gold_update_ts")
    )
    .dropDuplicates(["order_id"])
    .withColumn(
        "payment_completion_ratio",
        f.when(
            f.col("order_amount") > 0,
            f.col("paid_amount") / f.col("order_amount")
        ).otherwise(f.lit(0.0))
    )
    .withColumn(
        "payment_state",
        f.when(f.col("order_amount")==0, "Invalid_order_amount")
            .when(f.col("payment_completion_ratio") ==0, "Unpaid")
            .when(f.col("payment_completion_ratio") ==1, "Paid")
            .when(f.col("payment_completion_ratio") <1, "Partially_paid")
            .when(f.col("payment_completion_ratio") >1, "Overpaid")
    )
    .withColumn("gold_updated_at", f.to_timestamp(f.col("gold_update_ts")))
    .withColumn("gold_run_id", f.lit(gold_run_id))
)

print("gold delta rows ", gold_delta.count())
display(gold_delta)

## Step 7 -- Merge Gold current-state table

If Gold delta contains rows, this cell merges them into `gold_schema.orders_information`

If there are no impacted rows, nothing is merged.

In [0]:
if gold_delta.count()> 0:
  upsert_to_gold(gold_delta, "ecomdatabricks.gold_schema.orders_information", "order_id")
else:
  print("No new rows to insert in gold tables")

In [0]:
%sql
select * from ecomdatabricks.gold_schema.orders_information

## Step 8 -- Maintain Gold SCD Type 2 History
This cell updates the SCD2 history table

If a current Gold row changes, the old version is closed (`is_current=flase`) and a new current version is inserted

In [0]:
if not spark.catalog.tableExists("ecomdatabricks.gold_schema.orders_information_scd2"):
  spark.sql("""
            create table ecomdatabricks.gold_schema.orders_information_scd2
            using delta
            as
            select *,
            cast( null as timestamp) as valid_from_ts,
            cast( null as timestamp) as valid_to_ts,
            true as is_current
            from ecomdatabricks.gold_schema.orders_information
            where 1 = 0
          """)
  


if gold_delta.count() > 0:
  gold_delta.createOrReplaceTempView("gold_delta_view")
  spark.sql("""
            merge into ecomdatabricks.gold_schema.orders_information_scd2 as t
            using gold_delta_view as s
            on t.order_id = s.order_id and t.is_current = true
            when matched and (
              not(t.order_status <=>s.order_status) or
              not(t.order_amount <=> s.order_amount) or
              not(t.paid_amount <=> s.paid_amount) or
              not(t.payment_id <=> s.payment_id) or
              not(t.category <=> s.category) or
              not(t.product_name <=> s.product_name) or
              not(t.product_price <=> s.product_price)
            )
            then update set
              is_current = false,
              valid_to_ts = s.gold_updated_at
            
      """)
  
  spark.sql("""
            insert into ecomdatabricks.gold_schema.orders_information_scd2
            select s.*,
              s.gold_updated_at as valid_from_ts,
              cast(null as timestamp) as valid_to_ts,
              true as is_current
            from gold_delta_view s
            left join ecomdatabricks.gold_schema.orders_information_scd2 t
            on s.order_id = t.order_id and t.is_current=true
            where t.order_id is null or (
              not(t.order_status <=>s.order_status) or
              not(t.order_amount <=> s.order_amount) or
              not(t.paid_amount <=> s.paid_amount) or
              not(t.payment_id <=> s.payment_id) or
              not(t.category <=> s.category) or
              not(t.product_name <=> s.product_name) or
              not(t.product_price <=> s.product_price)
            )
            
          """)

In [0]:
%sql
select * from ecomdatabricks.gold_schema.orders_information_scd2

## Step 9 -- Update category-level Gold Aggregation

This cell recalculates  category-level business metrics only for categories impacted in the current run, then merges them into the category performance Gold table

In [0]:
if gold_delta.count() > 0:
  impacted_categories = (
    gold_delta.select("category").filter(f.col("category").isNotNull()).distinct()
  )

  category_perf_delta = (
    spark.read.table("ecomdatabricks.gold_schema.orders_information")
    .join(impacted_categories, "category", "inner")
    .groupBy("category")
    .agg(
      f.countDistinct("order_id").alias("total_orders"),
      f.sum(
        f.when(f.col("order_amount")> 0, f.col("order_amount"))
        .otherwise(f.lit(0.0))
        ).alias("Gross_Merchandise_Value"),
      f.sum(
          f.when(f.col("paid_amount")> 0, f.col("paid_amount"))
        .otherwise(f.lit(0.0))
        ).alias("Total_Paid_Amount"),
      f.avg(f.col("payment_completion_ratio")).alias("Avg_payment_completion_ratio"),
      (
        f.sum(f.when(f.col("payment_status")=="FAILED", 1).otherwise(0))/f.count("*")
      ).alias("Payment_Failure_Rate")
      
    )
  )

  upsert_to_gold(category_perf_delta, "ecomdatabricks.gold_schema.category_performance", "category")

  display(category_perf_delta)

In [0]:
%sql
select * from ecomdatabricks.gold_schema.category_performance

## Step 10 -- Publish Gold Snapshot to Volume

this cell writes two kinds of outputs to a Databricks Volume

- **latest snapshot** -> overwritten every successful run
- **timestamped historical snapshots** -> a new folder for each successful run.

this is useful for audit, rollback and teaching demos

In [0]:
spark.sql("create volume if not exists ecomdatabricks.gold_schema.gold_snapshot_vol")

In [0]:
latest_order_path = (
    "/Volumes/ecomdatabricks/gold_schema/gold_snapshot_vol/gold_latest/orders_information"
)

latest_category_path = (
    "/Volumes/ecomdatabricks/gold_schema/gold_snapshot_vol/gold_latest/category_performance"
)

historical_order_path = f"/Volumes/ecomdatabricks/gold_schema/gold_snapshot_vol/gold_snapshots/orders_information/run_date={run_date_str}/run_ts_{run_ts_str}"

historical_category_path = f"/Volumes/ecomdatabricks/gold_schema/gold_snapshot_vol/gold_snapshots/category_performance/run_date={run_date_str}/run_ts_{run_ts_str}"



spark.read.table("ecomdatabricks.gold_schema.orders_information").write.mode("overwrite").format("parquet").save(latest_order_path)
spark.read.table("ecomdatabricks.gold_schema.category_performance").write.mode("overwrite").format("parquet").save(latest_category_path)

spark.read.table("ecomdatabricks.gold_schema.orders_information").write.mode("overwrite").format("parquet").save(historical_order_path)
spark.read.table("ecomdatabricks.gold_schema.category_performance").write.mode("overwrite").format("parquet").save(historical_category_path)


print("Latest Order Path ", latest_order_path)
print("Latest Category Path ", latest_category_path)
print("Historical Order Path ", historical_order_path)
print("Historical Category Path ", historical_category_path)

In [0]:
latest_silver_ts = silver_orders_current.agg(f.max("bronze_ingested_at").alias("mx")).collect()[0]["mx"]

latest_silver_run_id = (
    silver_orders_current
    .filter(f.col("bronze_ingested_at")==latest_silver_ts)
    .agg(f.max("silver_run_id").alias("mx"))
    .collect()[0]["mx"]
) if latest_silver_ts is not None else None

upsert_gold_control("orders_information", latest_silver_run_id, latest_silver_ts, gold_delta.count())
display(spark.table("ecomdatabricks.gold_schema.processing_control"))